In [1]:
import pandas as pd

# 1. Carregar os datasets originais
df_results = pd.read_csv('results.csv')
df_shootouts = pd.read_csv('shootouts.csv')

# 2. Converter a coluna 'date' de texto (string) para o formato datetime
df_results['date'] = pd.to_datetime(df_results['date'])
df_shootouts['date'] = pd.to_datetime(df_shootouts['date'])

# 3. Definir a data de corte (dia seguinte à final da Copa de 2018)
data_corte = pd.to_datetime('2018-07-16')

# 4. Aplicar o filtro nos dois DataFrames
df_results_filtrado = df_results[df_results['date'] >= data_corte].copy()
df_shootouts_filtrado = df_shootouts[df_shootouts['date'] >= data_corte].copy()

# 5. Ordenar cronologicamente para garantir a ordem correta
df_results_filtrado = df_results_filtrado.sort_values('date').reset_index(drop=True)
df_shootouts_filtrado = df_shootouts_filtrado.sort_values('date').reset_index(drop=True)

# Exibir os resultados para validação
print("--- RESUMO DO FILTRO ---")
print(f"Jogos originais: {len(df_results)} | Jogos após Copa 2018: {len(df_results_filtrado)}")
print(f"Disputas de pênaltis originais: {len(df_shootouts)} | Pênaltis após Copa 2018: {len(df_shootouts_filtrado)}")

# Mostrando as 3 primeiras linhas do dataset de resultados filtrado
display(df_results_filtrado.head(3))

--- RESUMO DO FILTRO ---
Jogos originais: 49378 | Jogos após Copa 2018: 7674
Disputas de pênaltis originais: 675 | Pênaltis após Copa 2018: 145


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,2018-07-22,Liberia,Sierra Leone,0.0,0.0,Friendly,Monrovia,Liberia,False
1,2018-07-27,Saint Barthélemy,Anguilla,1.0,2.0,Friendly,Saint-Jean,Saint Barthélemy,False
2,2018-07-28,Saint Barthélemy,British Virgin Islands,3.0,4.0,Friendly,Saint-Jean,Saint Barthélemy,False


In [2]:
# Fazer a junção (merge) dos dois DataFrames
df = pd.merge(
    df_results_filtrado,
    df_shootouts_filtrado,
    on=['date', 'home_team', 'away_team'],
    how='left'
)

# Exibir informações sobre o novo DataFrame para validar a junção
print("--- INFORMAÇÕES DO DATAFRAME FINAL (df) ---")
print(f"Total de jogos: {len(df)}")
print(f"Colunas do df: {list(df.columns)}")

# Mostrar um exemplo de um jogo que foi a penáltis para verificar se a junção funcionou bem
# Vamos filtrar onde a coluna 'winner' não é nula
jogos_com_penaltis = df[df['winner'].notna()]

print(f"\nTotal de jogos que foram a penáltis: {len(jogos_com_penaltis)}")
print("\nExemplo de jogos com penáltis:")
display(jogos_com_penaltis.head(3))

--- INFORMAÇÕES DO DATAFRAME FINAL (df) ---
Total de jogos: 7674
Colunas do df: ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral', 'winner', 'first_shooter']

Total de jogos que foram a penáltis: 145

Exemplo de jogos com penáltis:


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,winner,first_shooter
564,2019-01-20,Jordan,Vietnam,1.0,1.0,AFC Asian Cup,Dubai,United Arab Emirates,True,Vietnam,Vietnam
566,2019-01-21,Australia,Uzbekistan,0.0,0.0,AFC Asian Cup,Al Ain,United Arab Emirates,True,Australia,Australia
651,2019-03-23,Oman,Singapore,1.0,1.0,Friendly,Kuala Lumpur,Malaysia,True,Oman,NaN


In [3]:
df['tournament'].value_counts(dropna=False)

tournament
Friendly                                             1935
FIFA World Cup qualification                         1767
UEFA Nations League                                   658
African Cup of Nations qualification                  560
UEFA Euro qualification                               501
                                                     ... 
Mukuru 4 Nations                                        2
CONIFA World Cup qualification                          1
CONMEBOL–UEFA Cup of Champions                          1
South Asian Super Cup                                   1
Diamond Jubilee International Football Tournament       1
Name: count, Length: 72, dtype: int64

In [4]:
# 1. Definir a Lista Branca de competições relevantes (oficiais de seleções principais)
# Não incluímos 'FIFA World Cup' e 'Friendly'
competicoes_relevantes = [
    # FIFA
    'FIFA World Cup',
    'FIFA World Cup qualification',

    # Europa
    'UEFA Euro',
    'UEFA Euro qualification',
    'UEFA Nations League',

    # América do Sul
    'Copa América',
    'Copa América qualification',

    # Concacaf (América do Norte/Central)
    'Gold Cup',
    'Gold Cup qualification',
    'CONCACAF Nations League',
    'CONCACAF Nations League qualification',

    # África
    'African Cup of Nations',
    'African Cup of Nations qualification',

    # Ásia
    'AFC Asian Cup',
    'AFC Asian Cup qualification',

    # Oceânia
    'Oceania Nations Cup',
    'Oceania Nations Cup qualification',

    # Intercontinentais oficiais
    'CONMEBOL-UEFA Cup of Champions' # Finalíssima
]

# 2. Aplicar o filtro no DataFrame
df_competitivo = df[df['tournament'].isin(competicoes_relevantes)].copy()

# 3. Validação do resultado
print("--- RESUMO DO FILTRO DE COMPETIÇÕES ---")
print(f"Total de jogos antes do filtro: {len(df)}")
print(f"Total de jogos estritamente competitivos: {len(df_competitivo)}")
print(f"Jogos descartados (Amistosos, Copa do Mundo e irrelevantes): {len(df) - len(df_competitivo)}")

# Verificar as competições que restaram
print("\nCompetições mantidas no dataset:")
df_competitivo['tournament'].value_counts()

--- RESUMO DO FILTRO DE COMPETIÇÕES ---
Total de jogos antes do filtro: 7674
Total de jogos estritamente competitivos: 4889
Jogos descartados (Amistosos, Copa do Mundo e irrelevantes): 2785

Competições mantidas no dataset:


tournament
FIFA World Cup qualification             1767
UEFA Nations League                       658
African Cup of Nations qualification      560
UEFA Euro qualification                   501
CONCACAF Nations League                   422
African Cup of Nations                    208
FIFA World Cup                            136
Gold Cup                                  124
AFC Asian Cup qualification               105
AFC Asian Cup                             102
UEFA Euro                                 102
Copa América                               86
CONCACAF Nations League qualification      68
Gold Cup qualification                     32
Oceania Nations Cup                        13
Oceania Nations Cup qualification           3
Copa América qualification                  2
Name: count, dtype: int64

In [5]:
import numpy as np

print("A calcular a Forma das Seleções (últimos 5 jogos)...")

# 1. Garantir ordem cronológica e criar um ID para o jogo
df_competitivo = df_competitivo.sort_values('date').reset_index(drop=True)
df_competitivo['match_id'] = df_competitivo.index

# 2. Definir regras de pontos (Vitória=3, Empate=1, Derrota=0)
condicoes = [
    df_competitivo['home_score'] > df_competitivo['away_score'],  # Casa vence
    df_competitivo['home_score'] == df_competitivo['away_score']  # Empate
]
pontos_casa = [3, 1]
pontos_fora = [0, 1]

# Aplicar os pontos gerados em cada jogo
df_competitivo['home_pts'] = np.select(condicoes, pontos_casa, default=0)
df_competitivo['away_pts'] = np.select(condicoes, pontos_fora, default=3)

# 3. Criar visão temporária para o Pandas processar o histórico por equipa
df_home_temp = df_competitivo[['match_id', 'date', 'home_team', 'home_pts']].rename(
    columns={'home_team': 'team', 'home_pts': 'pts'}
)
df_home_temp['is_home'] = True

df_away_temp = df_competitivo[['match_id', 'date', 'away_team', 'away_pts']].rename(
    columns={'away_team': 'team', 'away_pts': 'pts'}
)
df_away_temp['is_home'] = False

# Juntar tudo e ordenar por data
df_historico = pd.concat([df_home_temp, df_away_temp]).sort_values('date')

# 4. Calcular a soma dos últimos 5 jogos usando shift(1) para evitar Data Leakage
df_historico['form_pts'] = df_historico.groupby('team')['pts'].transform(
    lambda x: x.shift(1).rolling(window=5, min_periods=1).sum()
)

# Preencher NaN (o primeiro jogo da equipa na nossa base) com 0 pontos
df_historico['form_pts'] = df_historico['form_pts'].fillna(0)

# 5. Separar de volta para Casa e Fora
home_form = df_historico[df_historico['is_home'] == True][['match_id', 'form_pts']].rename(
    columns={'form_pts': 'home_form_pts'}
)
away_form = df_historico[df_historico['is_home'] == False][['match_id', 'form_pts']].rename(
    columns={'form_pts': 'away_form_pts'}
)

# 6. Juntar ao DataFrame principal e limpar as colunas temporárias
df_competitivo = df_competitivo.merge(home_form, on='match_id', how='left')
df_competitivo = df_competitivo.merge(away_form, on='match_id', how='left')

df_competitivo = df_competitivo.drop(columns=['match_id', 'home_pts', 'away_pts'])

print("✅ Forma calculada! 'home_form_pts' e 'away_form_pts' adicionadas.")

# Visualizar o resultado
display(df_competitivo[['date', 'home_team', 'away_team', 'home_form_pts', 'away_form_pts']].tail())

A calcular a Forma das Seleções (últimos 5 jogos)...
✅ Forma calculada! 'home_form_pts' e 'away_form_pts' adicionadas.


,date,home_team,away_team,home_form_pts,away_form_pts
4884,2026-06-27,DR Congo,Uzbekistan,12.0,8.0
4885,2026-06-27,Colombia,Portugal,10.0,4.0
4886,2026-06-27,Panama,England,10.0,9.0
4887,2026-06-27,Algeria,Austria,12.0,7.0
4888,2026-06-27,Croatia,Ghana,15.0,12.0


In [6]:
print("A calcular o Aproveitamento de Mandos sensível ao contexto (Casa, Fora e Neutro)...")

# 1. Garantir o ID da partida e os pontos
df_competitivo['match_id'] = df_competitivo.index
condicoes = [
    df_competitivo['home_score'] > df_competitivo['away_score'],
    df_competitivo['home_score'] == df_competitivo['away_score']
]
df_competitivo['home_pts_temp'] = np.select(condicoes, [3, 1], default=0)
df_competitivo['away_pts_temp'] = np.select(condicoes, [0, 1], default=3)

# 2. Criar a visão "longa" e classificar o tipo de mando daquela equipa naquele jogo
df_home = df_competitivo[['match_id', 'date', 'home_team', 'home_pts_temp', 'neutral']].copy()
df_home.columns = ['match_id', 'date', 'team', 'pts', 'neutral']
df_home['venue_type'] = np.where(df_home['neutral'] == True, 'Neutro', 'Casa')
df_home['is_home_team'] = True

df_away = df_competitivo[['match_id', 'date', 'away_team', 'away_pts_temp', 'neutral']].copy()
df_away.columns = ['match_id', 'date', 'team', 'pts', 'neutral']
df_away['venue_type'] = np.where(df_away['neutral'] == True, 'Neutro', 'Fora')
df_away['is_home_team'] = False

# 3. Juntar as visões e ordenar cronologicamente
df_venue = pd.concat([df_home, df_away]).sort_values('date')

# 4. A MÁGICA: Calcular o aproveitamento agrupando por "Equipa + Tipo de Mando"
# A função expanding() pega todo o histórico desde o início da base até ao jogo anterior (shift)
df_venue['pts_conquistados'] = df_venue.groupby(['team', 'venue_type'])['pts'].transform(
    lambda x: x.shift(1).expanding().sum()
)
df_venue['jogos_disputados'] = df_venue.groupby(['team', 'venue_type'])['pts'].transform(
    lambda x: x.shift(1).expanding().count()
)

# Cálculo do Aproveitamento: (Pontos Conquistados / Pontos Possíveis) * 100
# np.where previne erro de divisão por zero no primeiro jogo da história da seleção
df_venue['aproveitamento_mando'] = np.where(
    df_venue['jogos_disputados'] > 0,
    (df_venue['pts_conquistados'] / (df_venue['jogos_disputados'] * 3)) * 100,
    0
).round(1)

# 5. Separar os dados para as equipas 1 e 2 do nosso df original
home_stats = df_venue[df_venue['is_home_team'] == True][['match_id', 'venue_type', 'aproveitamento_mando']]
home_stats.columns = ['match_id', 'home_venue_type', 'home_aproveitamento']

away_stats = df_venue[df_venue['is_home_team'] == False][['match_id', 'venue_type', 'aproveitamento_mando']]
away_stats.columns = ['match_id', 'away_venue_type', 'away_aproveitamento']

# 6. Mesclar no DataFrame principal e limpar as colunas temporárias
df_competitivo = df_competitivo.merge(home_stats, on='match_id', how='left')
df_competitivo = df_competitivo.merge(away_stats, on='match_id', how='left')

df_competitivo = df_competitivo.drop(columns=['match_id', 'home_pts_temp', 'away_pts_temp'])

print("✅ Aproveitamento de mandos calculado com sucesso (escala 0 a 100%)!")

# Mostrar o resultado das últimas partidas para validação visual
cols_view = ['date', 'home_team', 'away_team', 'neutral', 'home_venue_type', 'home_aproveitamento', 'away_venue_type', 'away_aproveitamento']
display(df_competitivo[cols_view].tail(5))

A calcular o Aproveitamento de Mandos sensível ao contexto (Casa, Fora e Neutro)...
✅ Aproveitamento de mandos calculado com sucesso (escala 0 a 100%)!


,date,home_team,away_team,neutral,home_venue_type,home_aproveitamento,away_venue_type,away_aproveitamento
4884,2026-06-27,DR Congo,Uzbekistan,True,Neutro,57.6,Neutro,69.2
4885,2026-06-27,Colombia,Portugal,True,Neutro,64.8,Neutro,45.8
4886,2026-06-27,Panama,England,True,Neutro,53.6,Neutro,52.6
4887,2026-06-27,Algeria,Austria,True,Neutro,68.0,Neutro,55.6
4888,2026-06-27,Croatia,Ghana,True,Neutro,47.9,Neutro,38.1


In [7]:
print("A calcular o Intervalo de Descanso (Dias desde o último jogo)...")

# 1. Garantir que temos o ID da partida
df_competitivo['match_id'] = df_competitivo.index

# 2. Criar a visão "longa" apenas com o ID, Data e Nome da Equipa
df_home_date = df_competitivo[['match_id', 'date', 'home_team']].copy()
df_home_date.columns = ['match_id', 'date', 'team']
df_home_date['is_home_team'] = True

df_away_date = df_competitivo[['match_id', 'date', 'away_team']].copy()
df_away_date.columns = ['match_id', 'date', 'team']
df_away_date['is_home_team'] = False

# Juntar as duas visões e ordenar cronologicamente
df_datas = pd.concat([df_home_date, df_away_date]).sort_values('date')

# 3. Encontrar a data do jogo anterior de cada equipa
df_datas['prev_date'] = df_datas.groupby('team')['date'].shift(1)

# 4. Calcular a diferença exata em dias
df_datas['rest_days'] = (df_datas['date'] - df_datas['prev_date']).dt.days
df_datas['rest_days'] = df_datas['rest_days'].fillna(90)

# 5. Separar os dados para as equipas 1 e 2 do nosso df original
home_stats = df_datas[df_datas['is_home_team'] == True][['match_id', 'rest_days']]
home_stats.columns = ['match_id', 'home_rest_days']

away_stats = df_datas[df_datas['is_home_team'] == False][['match_id', 'rest_days']]
away_stats.columns = ['match_id', 'away_rest_days']

# 6. Mesclar no DataFrame principal
df_competitivo = df_competitivo.merge(home_stats, on='match_id', how='left')
df_competitivo = df_competitivo.merge(away_stats, on='match_id', how='left')

# 7. Calcular a diferença de descanso e remover colunas auxiliares
# Positivo: mandante mais descansado. Negativo: visitante mais descansado.
df_competitivo['rest_days_diff'] = df_competitivo['home_rest_days'] - df_competitivo['away_rest_days']
df_competitivo = df_competitivo.drop(columns=['match_id', 'home_rest_days', 'away_rest_days'])

print("✅ Diferença de dias de descanso calculada ('rest_days_diff').")

# Visualizar o resultado prático para validar
cols_view = ['date', 'home_team', 'away_team', 'rest_days_diff']
display(df_competitivo[cols_view].tail(5))

A calcular o Intervalo de Descanso (Dias desde o último jogo)...
✅ Diferença de dias de descanso calculada ('rest_days_diff').


,date,home_team,away_team,rest_days_diff
4884,2026-06-27,DR Congo,Uzbekistan,0.0
4885,2026-06-27,Colombia,Portugal,0.0
4886,2026-06-27,Panama,England,0.0
4887,2026-06-27,Algeria,Austria,0.0
4888,2026-06-27,Croatia,Ghana,0.0


In [8]:
print("A calcular o Histórico de Confronto Direto (H2H) - Últimos 5 jogos entre as equipas...")

# 1. Garantir o ID e os pontos do jogo atual (reaproveitando a lógica)
df_competitivo['match_id'] = df_competitivo.index
condicoes = [
    df_competitivo['home_score'] > df_competitivo['away_score'],
    df_competitivo['home_score'] == df_competitivo['away_score']
]
df_competitivo['home_pts_temp'] = np.select(condicoes, [3, 1], default=0)
df_competitivo['away_pts_temp'] = np.select(condicoes, [0, 1], default=3)

# 2. Criar a visão "longa", mas agora mapeando QUEM é o oponente
# Perspetiva da equipa da Casa
df_home_h2h = df_competitivo[['match_id', 'date', 'home_team', 'away_team', 'home_pts_temp']].copy()
df_home_h2h.columns = ['match_id', 'date', 'team', 'opponent', 'pts']
df_home_h2h['is_home_team'] = True

# Perspetiva da equipa de Fora
df_away_h2h = df_competitivo[['match_id', 'date', 'away_team', 'home_team', 'away_pts_temp']].copy()
df_away_h2h.columns = ['match_id', 'date', 'team', 'opponent', 'pts']
df_away_h2h['is_home_team'] = False

# 3. Juntar e ordenar cronologicamente
df_h2h = pd.concat([df_home_h2h, df_away_h2h]).sort_values('date')

# 4. A MÁGICA: Agrupar por [Equipa + Oponente]
# Assim, o Pandas filtra o histórico apenas para os jogos exatos entre essas duas seleções
df_h2h['h2h_form_pts'] = df_h2h.groupby(['team', 'opponent'])['pts'].transform(
    lambda x: x.shift(1).rolling(window=5, min_periods=1).sum()
)

# Se é a primeira vez que se enfrentam na nossa base (ex: Brasil x Islândia), preenchemos com 0
df_h2h['h2h_form_pts'] = df_h2h['h2h_form_pts'].fillna(0)

# 5. Separar de volta para o formato de Casa e Fora
home_stats = df_h2h[df_h2h['is_home_team'] == True][['match_id', 'h2h_form_pts']]
home_stats.columns = ['match_id', 'home_h2h_pts']

away_stats = df_h2h[df_h2h['is_home_team'] == False][['match_id', 'h2h_form_pts']]
away_stats.columns = ['match_id', 'away_h2h_pts']

# 6. Mesclar no DataFrame principal e limpar as colunas de apoio
df_competitivo = df_competitivo.merge(home_stats, on='match_id', how='left')
df_competitivo = df_competitivo.merge(away_stats, on='match_id', how='left')

df_competitivo = df_competitivo.drop(columns=['match_id', 'home_pts_temp', 'away_pts_temp'])

print("✅ H2H (Confronto Direto) calculado!")

# Vamos validar procurando um jogo clássico recorrente, como Argentina x Chile
clash_mask = (df_competitivo['home_team'].isin(['Argentina', 'Chile'])) & (df_competitivo['away_team'].isin(['Argentina', 'Chile']))
cols_view = ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'home_h2h_pts', 'away_h2h_pts']

display(df_competitivo[clash_mask][cols_view].tail(5))

A calcular o Histórico de Confronto Direto (H2H) - Últimos 5 jogos entre as equipas...
✅ H2H (Confronto Direto) calculado!


,date,home_team,away_team,home_score,away_score,home_h2h_pts,away_h2h_pts
1526,2021-06-14,Argentina,Chile,1.0,1.0,4.0,1.0
2114,2022-01-27,Chile,Argentina,1.0,2.0,2.0,5.0
3565,2024-06-25,Chile,Argentina,0.0,1.0,2.0,8.0
3635,2024-09-05,Argentina,Chile,3.0,0.0,11.0,2.0
4290,2025-06-05,Chile,Argentina,0.0,1.0,2.0,11.0


In [9]:
clash_mask = (df_competitivo['home_team'].isin(['Argentina', 'Chile'])) & (df_competitivo['away_team'].isin(['Argentina', 'Chile']))
cols_view = ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'home_h2h_pts', 'away_h2h_pts']

display(df_competitivo[clash_mask][cols_view])

,date,home_team,away_team,home_score,away_score,home_h2h_pts,away_h2h_pts
580,2019-07-06,Argentina,Chile,2.0,1.0,0.0,0.0
1434,2021-06-03,Argentina,Chile,1.0,1.0,3.0,0.0
1526,2021-06-14,Argentina,Chile,1.0,1.0,4.0,1.0
2114,2022-01-27,Chile,Argentina,1.0,2.0,2.0,5.0
3565,2024-06-25,Chile,Argentina,0.0,1.0,2.0,8.0
3635,2024-09-05,Argentina,Chile,3.0,0.0,11.0,2.0
4290,2025-06-05,Chile,Argentina,0.0,1.0,2.0,11.0


In [10]:
print("A calcular a Média de Saldo de Gols (últimos 5 jogos)...")

# 1. Garantir o ID da partida
df_competitivo['match_id'] = df_competitivo.index

# 2. Calcular o Saldo de Gols (GD - Goal Difference) da partida específica
df_competitivo['home_gd_match'] = df_competitivo['home_score'] - df_competitivo['away_score']
df_competitivo['away_gd_match'] = df_competitivo['away_score'] - df_competitivo['home_score']

# 3. Criar a visão "longa" para o Pandas processar o histórico por equipa
df_home_gd = df_competitivo[['match_id', 'date', 'home_team', 'home_gd_match']].copy()
df_home_gd.columns = ['match_id', 'date', 'team', 'gd_match']
df_home_gd['is_home_team'] = True

df_away_gd = df_competitivo[['match_id', 'date', 'away_team', 'away_gd_match']].copy()
df_away_gd.columns = ['match_id', 'date', 'team', 'gd_match']
df_away_gd['is_home_team'] = False

# Juntar tudo e ordenar cronologicamente
df_historico_gd = pd.concat([df_home_gd, df_away_gd]).sort_values('date')

# 4. A MÁGICA: Calcular a MÉDIA do saldo de gols dos últimos 5 jogos
# Usamos shift(1) para não incluir o jogo atual (evitar Data Leakage)
df_historico_gd['form_gd'] = df_historico_gd.groupby('team')['gd_match'].transform(
    lambda x: x.shift(1).rolling(window=5, min_periods=1).mean()
)

# Preencher NaN (o primeiro jogo da equipa na base) com 0 de saldo
df_historico_gd['form_gd'] = df_historico_gd['form_gd'].fillna(0).round(2)

# 5. Separar de volta para Casa e Fora
home_stats = df_historico_gd[df_historico_gd['is_home_team'] == True][['match_id', 'form_gd']]
home_stats.columns = ['match_id', 'home_form_gd']

away_stats = df_historico_gd[df_historico_gd['is_home_team'] == False][['match_id', 'form_gd']]
away_stats.columns = ['match_id', 'away_form_gd']

# 6. Mesclar no DataFrame principal e limpar as colunas temporárias
df_competitivo = df_competitivo.merge(home_stats, on='match_id', how='left')
df_competitivo = df_competitivo.merge(away_stats, on='match_id', how='left')

df_competitivo = df_competitivo.drop(columns=['match_id', 'home_gd_match', 'away_gd_match'])

print("✅ Média de Saldo de Gols calculada! 'home_form_gd' e 'away_form_gd' adicionadas.")

# Visualizar o resultado para validação
cols_view = ['date', 'home_team', 'away_team', 'home_form_gd', 'away_form_gd']
display(df_competitivo[cols_view].tail(5))

A calcular a Média de Saldo de Gols (últimos 5 jogos)...
✅ Média de Saldo de Gols calculada! 'home_form_gd' e 'away_form_gd' adicionadas.


,date,home_team,away_team,home_form_gd,away_form_gd
4884,2026-06-27,DR Congo,Uzbekistan,1.00,1.00
4885,2026-06-27,Colombia,Portugal,2.00,2.00
4886,2026-06-27,Panama,England,1.33,3.00
4887,2026-06-27,Algeria,Austria,0.33,0.33
4888,2026-06-27,Croatia,Ghana,2.00,2.33


In [11]:
print("A calcular o Histórico de Aproveitamento em Penáltis...")

# 1. Garantir o ID da partida
df_competitivo['match_id'] = df_competitivo.index

# 2. Criar visões separadas mapeando se o jogo foi a penáltis e quem ganhou
# Para a equipa da Casa
df_home_pen = df_competitivo[['match_id', 'date', 'home_team', 'winner']].copy()
df_home_pen.columns = ['match_id', 'date', 'team', 'winner']
df_home_pen['went_to_pen'] = df_home_pen['winner'].notna() # True se teve penáltis
df_home_pen['won_pen'] = df_home_pen['winner'] == df_home_pen['team'] # True se o time ganhou os penáltis
df_home_pen['is_home_team'] = True

# Para a equipa de Fora
df_away_pen = df_competitivo[['match_id', 'date', 'away_team', 'winner']].copy()
df_away_pen.columns = ['match_id', 'date', 'team', 'winner']
df_away_pen['went_to_pen'] = df_away_pen['winner'].notna()
df_away_pen['won_pen'] = df_away_pen['winner'] == df_away_pen['team']
df_away_pen['is_home_team'] = False

# Juntar tudo e ordenar cronologicamente
df_penalties = pd.concat([df_home_pen, df_away_pen]).sort_values('date')

# 3. Converter para números (0 e 1) para facilitar a soma cumulativa
df_penalties['pen_disputado'] = df_penalties['went_to_pen'].astype(int)
df_penalties['pen_vencido'] = (df_penalties['won_pen'] & df_penalties['went_to_pen']).astype(int)

# 4. Calcular o acumulado histórico (shift(1) garante que o jogo do dia não conta)
df_penalties['hist_pen_disputados'] = df_penalties.groupby('team')['pen_disputado'].transform(
    lambda x: x.shift(1).expanding().sum()
).fillna(0)

df_penalties['hist_pen_vencidos'] = df_penalties.groupby('team')['pen_vencido'].transform(
    lambda x: x.shift(1).expanding().sum()
).fillna(0)

# 5. Calcular a Taxa de Vitória em Penáltis
# Usamos np.where para definir 50% (0.5) para quem nunca disputou penáltis na base
df_penalties['penalties_win_rate'] = np.where(
    df_penalties['hist_pen_disputados'] > 0,
    df_penalties['hist_pen_vencidos'] / df_penalties['hist_pen_disputados'],
    0.5
).round(3)

# 6. Separar de volta para Casa e Fora
home_stats = df_penalties[df_penalties['is_home_team'] == True][['match_id', 'penalties_win_rate']]
home_stats.columns = ['match_id', 'home_pen_win_rate']

away_stats = df_penalties[df_penalties['is_home_team'] == False][['match_id', 'penalties_win_rate']]
away_stats.columns = ['match_id', 'away_pen_win_rate']

# 7. Mesclar no DataFrame principal e limpar
df_competitivo = df_competitivo.merge(home_stats, on='match_id', how='left')
df_competitivo = df_competitivo.merge(away_stats, on='match_id', how='left')
df_competitivo = df_competitivo.drop(columns=['match_id'])

print("✅ Histórico de Penáltis calculado! 'home_pen_win_rate' e 'away_pen_win_rate' adicionadas.")

# Visualização para validar (filtrando um jogo que sabemos que foi para pênaltis)
jogos_com_penaltis = df_competitivo[df_competitivo['winner'].notna()]
cols_view = ['date', 'home_team', 'away_team', 'winner', 'home_pen_win_rate', 'away_pen_win_rate']
display(jogos_com_penaltis[cols_view].tail())

A calcular o Histórico de Aproveitamento em Penáltis...
✅ Histórico de Penáltis calculado! 'home_pen_win_rate' e 'away_pen_win_rate' adicionadas.


,date,home_team,away_team,winner,home_pen_win_rate,away_pen_win_rate
4799,2026-01-17,Egypt,Nigeria,Nigeria,0.4,0.333
4805,2026-03-26,Czech Republic,Republic of Ireland,Czech Republic,0.5,0.000
4810,2026-03-26,Wales,Bosnia and Herzegovina,Bosnia and Herzegovina,0.0,0.000
4811,2026-03-31,Czech Republic,Denmark,Czech Republic,1.0,0.500
4816,2026-03-31,Bosnia and Herzegovina,Italy,Bosnia and Herzegovina,0.5,1.000


In [12]:
df_competitivo.isna().sum()

date                      0
home_team                 0
away_team                 0
home_score               72
away_score               72
tournament                0
city                      0
country                   0
neutral                   0
winner                 4806
first_shooter          4808
home_form_pts             0
away_form_pts             0
home_venue_type           0
home_aproveitamento       0
away_venue_type           0
away_aproveitamento       0
rest_days_diff            0
home_h2h_pts              0
away_h2h_pts              0
home_form_gd              0
away_form_gd              0
home_pen_win_rate         0
away_pen_win_rate         0
dtype: int64

In [13]:
# 1. Drop nas linhas com score nulo
linhas_antes = len(df_competitivo)
df_competitivo = df_competitivo.dropna(subset=['home_score', 'away_score'])
print(f"✅ {linhas_antes - len(df_competitivo)} linhas com score nulo removidas.")

# 2. winner nulo = jogo sem disputa de pênaltis — preencher explicitamente
df_competitivo['winner'] = df_competitivo['winner'].fillna('No shootout')

# 3. first_shooter: preencher apenas os nulls correspondentes aos jogos sem pênaltis
#    (os 4806 que já foram tratados no winner)
mask_no_shootout = df_competitivo['winner'] == 'No shootout'
df_competitivo.loc[mask_no_shootout, 'first_shooter'] = (
    df_competitivo.loc[mask_no_shootout, 'first_shooter'].fillna('No shootout')
)

# 4. Drop dos 2 registros extras onde first_shooter ainda é nulo
#    (jogos que tiveram pênaltis mas first_shooter não foi registrado)
linhas_antes = len(df_competitivo)
df_competitivo = df_competitivo.dropna(subset=['first_shooter'])
print(f"✅ {linhas_antes - len(df_competitivo)} registros extras de first_shooter removidos.")

print(f"\nNulos restantes:")
print(df_competitivo[['home_score', 'away_score', 'winner', 'first_shooter']].isnull().sum())

✅ 72 linhas com score nulo removidas.
✅ 2 registros extras de first_shooter removidos.

Nulos restantes:
home_score       0
away_score       0
winner           0
first_shooter    0
dtype: int64


In [14]:
print("A calcular o peso dos campeonatos...")

# 1. Mapeamento de peso por tipo de competição
pesos_torneio = {
    # FIFA
    'FIFA World Cup'                        : 5.0,
    'FIFA World Cup qualification'          : 3.0,

    # Europa
    'UEFA Euro'                             : 4.5,
    'UEFA Euro qualification'               : 3.0,
    'UEFA Nations League'                   : 2.5,

    # América do Sul
    'Copa América'                          : 4.5,
    'Copa América qualification'            : 3.0,

    # Concacaf
    'Gold Cup'                              : 3.5,
    'Gold Cup qualification'                : 2.5,
    'CONCACAF Nations League'               : 2.0,
    'CONCACAF Nations League qualification' : 1.5,

    # África
    'African Cup of Nations'                : 4.0,
    'African Cup of Nations qualification'  : 2.5,

    # Ásia
    'AFC Asian Cup'                         : 4.0,
    'AFC Asian Cup qualification'           : 2.5,

    # Oceânia
    'Oceania Nations Cup'                   : 3.5,
    'Oceania Nations Cup qualification'     : 2.5,

    # Intercontinentais oficiais
    'CONMEBOL-UEFA Cup of Champions'        : 3.0,
}

# 2. Aplicar os pesos — torneios não mapeados recebem peso neutro (1.0)
df_competitivo['tournament_weight'] = (
    df_competitivo['tournament']
    .map(pesos_torneio)
    .fillna(1.0)
)

print("✅ Pesos dos campeonatos calculados ('tournament_weight').")

# 3. Verificar se algum torneio do dataset ficou fora do mapeamento
torneios_no_dataset = set(df_competitivo['tournament'].unique())
torneios_mapeados = set(pesos_torneio.keys())
torneios_nao_mapeados = torneios_no_dataset - torneios_mapeados

if torneios_nao_mapeados:
    print(f"\n⚠️ {len(torneios_nao_mapeados)} torneios receberam peso padrão (1.0):")
    for t in sorted(torneios_nao_mapeados):
        print(f"  - {t}")
else:
    print("\n✅ Todos os torneios do dataset foram mapeados explicitamente.")

A calcular o peso dos campeonatos...
✅ Pesos dos campeonatos calculados ('tournament_weight').

✅ Todos os torneios do dataset foram mapeados explicitamente.


In [15]:
df_competitivo.to_csv('jogos_tratado.csv',index=False)